In [1]:
import netCDF4 as nc
import matplotlib.pyplot as plt
from matplotlib.colors import from_levels_and_colors
import numpy as np
from matplotlib import animation
from matplotlib.colors import BoundaryNorm
import matplotlib.colors as cols
import matplotlib as mpl
from mpl_toolkits.basemap import Basemap
from matplotlib import ticker, cm
import seaborn as sns

In [2]:
# 目标路径：icepack/columnphysics/
# 核心文件：icepack_shortwave.F90（辐射与反照率计算）、
#         icepack_meltpond_cesm.F90（融池计算）、
#         icepack_therm_vertical.F90（垂直热力学）

In [3]:
# 目标路径：cicecore/cicedynB/dynamics/
# 核心文件：ice_dyn_evp.F90 （弹-黏-塑性求解器）
# 特点：涉及二维空间梯度与 Halo 通信，若用 CNN 替代，必须在调用 AI 推理前手动执行边界通信（ice_HaloUpdate）
#

In [4]:
##### fortran

module ice_ai_mod
  use ice_kinds_mod, only: dbl_kind, int_kind, char_len_long
  use ftorch, only: torch_model, torch_tensor, torch_model_load, &
                    torch_tensor_from_array, torch_model_forward, &
                    torch_tensor_delete, torch_model_delete, &
                    torch_kFloat32, torch_kCPU, torch_kCUDA
  implicit none
  private

  public :: init_cice_ai, run_cice_ai_albedo, finalize_cice_ai

  ! 全局保存已加载的 PyTorch 模型实例
  type(torch_model), save :: albedo_model
  logical, save :: is_ai_initialized = .false.

contains

  !-----------------------------------------------------------------------
  ! 1. 初始化阶段调用：加载 TorchScript (.pt) 模型
  !-----------------------------------------------------------------------
  subroutine init_cice_ai(model_path, use_gpu)
    character(len=*), intent(in) :: model_path
    logical, intent(in), optional :: use_gpu
    integer :: device_type

    if (is_ai_initialized) return

    device_type = torch_kCPU
    if (present(use_gpu)) then
      if (use_gpu) device_type = torch_kCUDA
    end if

    ! 从磁盘加载导出的 .pt 模型到内存/显存
    call torch_model_load(albedo_model, trim(model_path), device_type)
    is_ai_initialized = .true.
  end subroutine init_cice_ai

  !-----------------------------------------------------------------------
  ! 2. 时间步内调用：执行批量前向推理
  !-----------------------------------------------------------------------
  subroutine run_cice_ai_albedo(npts, Tsfc, hsno, coszen, apnd, alvdr, alidr)
    integer(kind=int_kind), intent(in) :: npts
    real(kind=dbl_kind), dimension(npts), intent(in)  :: Tsfc, hsno, coszen, apnd
    real(kind=dbl_kind), dimension(npts), intent(out) :: alvdr, alidr

    ! PyTorch 默认使用 float32，需做精度转换缓冲
    real(kind=4), dimension(npts, 4), target :: in_buf
    real(kind=4), dimension(npts, 2), target :: out_buf
    type(torch_tensor), dimension(1) :: in_tensors, out_tensors
    integer :: in_dims(2), out_dims(2)

    if (npts <= 0) return

    ! 组装输入特征矩阵 (Double -> Float32)
    in_buf(:, 1) = real(Tsfc,   kind=4)
    in_buf(:, 2) = real(hsno,   kind=4)
    in_buf(:, 3) = real(coszen, kind=4)
    in_buf(:, 4) = real(apnd,   kind=4)

    in_dims  = [npts, 4]
    out_dims = [npts, 2]

    ! 零拷贝绑定 Fortran 数组内存到 Torch Tensor
    call torch_tensor_from_array(in_tensors(1),  in_buf,  in_dims,  torch_kFloat32, torch_kCPU)
    call torch_tensor_from_array(out_tensors(1), out_buf, out_dims, torch_kFloat32, torch_kCPU)

    ! 执行推理
    call torch_model_forward(albedo_model, in_tensors, out_tensors)

    ! 提取结果并转回 CICE 双精度 (Float32 -> Double)
    alvdr(:) = real(out_buf(:, 1), kind=dbl_kind)
    alidr(:) = real(out_buf(:, 2), kind=dbl_kind)

    ! 释放 Tensor 封装指针（不释放底层 Fortran 数组）
    call torch_tensor_delete(in_tensors(1))
    call torch_tensor_delete(out_tensors(1))
  end subroutine run_cice_ai_albedo

  !-----------------------------------------------------------------------
  ! 3. 模式结束阶段调用：释放模型内存
  !-----------------------------------------------------------------------
  subroutine finalize_cice_ai()
    if (is_ai_initialized) then
      call torch_model_delete(albedo_model)
      is_ai_initialized = .false.
    end if
  end subroutine finalize_cice_ai

end module ice_ai_mod


SyntaxError: invalid syntax (813368437.py, line 3)

In [5]:
########## 打开 CICE_InitMod.F90（位于 cicecore/cicedynB/infrastructure/），在 cice_init 末尾加入：
use ice_ai_mod, only: init_cice_ai
...
! 建议将模型放在共享存储路径下，确保所有 MPI 进程都能读取
call init_cice_ai('/path/to/your/models/cice_albedo_net.pt', use_gpu=.false.)

SyntaxError: invalid syntax (1718796652.py, line 2)

In [6]:
## 在 icepack_shortwave.F90（或调用它的上层驱动 ice_step_mod.F90）中，
## 严禁在 (i, j, ncat) 的三重循环最内层逐点调用 PyTorch（函数调用开销极大）。
## 应当将当前 Block 内所有**有冰格点（aice > puny）**打包成一个 1D 批次一次性送入 PyTorch：
use ice_ai_mod, only: run_cice_ai_albedo
...
! 1. 统计当前 Block 内的有效海冰格点数 n_ice
! 2. 将有效格点的 Tsfc, hsno, coszen, apnd 压缩进 1D 数组 (1:n_ice)
call run_cice_ai_albedo(n_ice, Tsfc_1d, hsno_1d, coszen_1d, apnd_1d, alvdr_1d, alidr_1d)
! 3. 将预测出的 alvdr_1d, alidr_1d 解包填回 2D/3D 网格，无冰格点保持默认海洋值

SyntaxError: invalid syntax (3505350648.py, line 4)

In [7]:
## 在结束阶段释放资源
## 打开 CICE_FinalMod.F90，在 cice_finalize 中加入：
use ice_ai_mod, only: finalize_cice_ai
...
call finalize_cice_ai()

SyntaxError: invalid syntax (4032541481.py, line 3)